# 10 · Phiên nghe last.fm (ticket #55)

Biến 253 triệu lượt nghe last.fm (Music4All-Onion, chưa có khái niệm phiên) thành bảng phiên giống `sessions.parquet` của MPD, để dùng lại `consecutive_pairs`, `smoothness` và điểm mix học (#23) trên bộ thứ hai. Cách cắt ghi ở `src/data/lastfm.py` và `docs/data-contract.md` (mục Bộ phụ last.fm).

**Input** (Add Input): Kaggle Dataset `mixnext-onion` (output notebook 09).

**Output** (`/kaggle/working/out/lastfm/`):
- `sessions.parquet`: `session_id`, `user_id`, `split` (theo user), `pos`, `track_id`
- `lastfm_log.json`: phễu từng bước, độ nhạy theo khoảng nghỉ 15/30/60 phút, thống kê từng split

#56 thêm `tracks.parquet` vào cùng notebook này (Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó).

Chỉ cần CPU, bật **Internet** để clone repo. Nếu thiếu RAM: đặt `SAMPLE_USERS` < 1 (tỉ lệ ghi vào log).

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #55 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
ONION = sorted(Path("/kaggle/input").rglob("listening_events.parquet"))[0].parent
OUT = Path("/kaggle/working/out/lastfm")
SAMPLE_USERS = 1.0   # giảm (ví dụ 0.5) nếu hết RAM
print("input:", ONION, sorted(p.name for p in ONION.iterdir()))

In [ ]:
import json, time
from src.data.lastfm import main

t0 = time.time()
log = main(["--onion", str(ONION), "--out", str(OUT), "--sample-users", str(SAMPLE_USERS)])
print(f"{time.time() - t0:.0f} giây")

In [ ]:
# Phễu, độ nhạy theo khoảng nghỉ, các split
import pandas as pd
print(pd.DataFrame(log["funnel"]).to_string(index=False))
print()
print(pd.DataFrame(log["sensitivity"]).T.rename_axis("gap (phút)").to_string())
print()
print(pd.DataFrame(log["splits"]).T.to_string())
sessions = pd.read_parquet(OUT / "sessions.parquet")
lengths = sessions.groupby("session_id").size()
print(f"\nđộ dài phiên: trung vị {lengths.median():.0f}, p90 {lengths.quantile(0.9):.0f}, tối đa {lengths.max()}")

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên **`mixnext-lastfm`**, **Visibility: Private**, License CC BY 4.0 (dữ liệu gốc Music4All-Onion).

## Kết quả (Kaggle CPU, 28/11/2026, toàn bộ user, `SAMPLE_USERS` = 1)

| bước | lượt | phiên | bài | user |
|---|---:|---:|---:|---:|
| đầu vào | 252.984.396 | | 56.512 | 119.140 |
| bỏ timestamp hỏng (trước 2002) | 252.984.395 | | 56.512 | 119.140 |
| cắt theo khoảng nghỉ 30 phút | 252.984.395 | 80.178.838 | 56.512 | 119.140 |
| bỏ bài không có đặc trưng | 252.860.043 | 80.162.092 | 56.463 | 119.140 |
| gộp lặp liên tiếp | 234.774.072 | 80.162.092 | 56.463 | 119.140 |
| giữ phiên ≥ 5 bài | **119.447.177** | **13.292.849** | 56.432 | 105.212 |

| khoảng nghỉ | phiên | cặp liền nhau |
|---:|---:|---:|
| 15 phút | 9.889.636 | 68.330.510 |
| **30 phút** | **13.292.849** | **106.154.328** |
| 60 phút | 14.602.312 | 133.485.717 |

| split | user | phiên | lượt |
|---|---:|---:|---:|
| train | 84.170 | 10.589.365 | 94.953.715 |
| val | 10.521 | 1.358.159 | 12.183.129 |
| test | 10.521 | 1.345.325 | 12.310.333 |

Độ dài phiên: trung vị 7, p90 14, tối đa **30.830**.

- Chỉ 1 lượt có timestamp hỏng (1970). Essentia phủ gần hết: bỏ bài không có đặc trưng chỉ mất 124 nghìn lượt (0,05%).
- Gộp lặp bỏ 18 triệu lượt (7%): người nghe last.fm hay phát lại một bài nhiều lần liền.
- Bước giữ phiên ≥ 5 bài bỏ nhiều nhất (80 triệu phiên → 13,3 triệu): phần lớn phiên rất ngắn. Còn 106 triệu cặp liền nhau để học điểm mix, lớn hơn nhiều so với MPD.
- Số cặp tăng mạnh theo khoảng nghỉ (68 → 106 → 133 triệu cho 15 → 30 → 60 phút): ngưỡng 30 phút là một lựa chọn, báo cáo nên nêu.
- Chia theo user đúng 80/10/10 về số user.
- **Lưu ý cho #57:** có phiên dài bất thường (tối đa 30.830 bài, nghe liên tục nhiều ngày không nghỉ quá 30 phút), có thể là phần mềm tự scrobble hoặc radio bật suốt. Khi bootstrap theo phiên, những phiên này nặng ngang hàng nghìn phiên thường; #57 nên cân nhắc giới hạn độ dài phiên hoặc báo cả kết quả khi bỏ chúng.